In [12]:
import pandas as pd
import numpy as np

In [72]:
dl_data = pd.read_excel('5G_DL.xlsx', sheet_name='Series Formatted Data')
ul_data = pd.read_excel('5G_UL.xlsx', sheet_name='Series Formatted Data')
scanner_data = pd.read_excel('5G_Scanner.xlsx', sheet_name='Series Formatted Data')

dl_data = dl_data.drop(["Message","Technology_Mode"],axis=1)
dl_data = dl_data.loc[:, ~dl_data.columns.str.contains('^Unnamed')]

ul_data = ul_data.drop(["Message","Technology_Mode"],axis=1)
ul_data = ul_data.loc[:, ~ul_data.columns.str.contains('^Unnamed')]

scanner_data = scanner_data.drop(["Message"],axis=1)
scanner_data = scanner_data.loc[:, ~scanner_data.columns.str.contains('^Unnamed')]

In [73]:
def scanner_group_by_time(data):
    new_data = pd.DataFrame()

    last_known_data_cols = ['Longitude', 'Latitude', 'NR_Scan_NR_ARFCN',
       'NR_Scan_PCI_SortedBy_RSRP_0', 'NR_Scan_PCI_SortedBy_RSRP_1',
       'NR_Scan_PCI_SortedBy_RSRP_2', 'NR_Scan_PCI_SortedBy_RSRP_3',
       'NR_Scan_PCI_SortedBy_RSRP_4', 'NR_Scan_PCI_SortedBy_RSRP_5',
       'NR_Scan_PCI_SortedBy_RSRP_6', 'NR_Scan_SSB_RSRP_SortedBy_RSRP_0',
       'NR_Scan_SSB_RSRP_SortedBy_RSRP_1', 'NR_Scan_SSB_RSRP_SortedBy_RSRP_2',
       'NR_Scan_SSB_RSRP_SortedBy_RSRP_3', 'NR_Scan_SSB_RSRP_SortedBy_RSRP_4',
       'NR_Scan_SSB_RSRP_SortedBy_RSRP_5', 'NR_Scan_SSB_RSRP_SortedBy_RSRP_6',
       'NR_Scan_SSB_RSRQ_SortedBy_RSRP_0', 'NR_Scan_SSB_RSRQ_SortedBy_RSRP_1',
       'NR_Scan_SSB_RSRQ_SortedBy_RSRP_2', 'NR_Scan_SSB_RSRQ_SortedBy_RSRP_3',
       'NR_Scan_SSB_RSRQ_SortedBy_RSRP_4', 'NR_Scan_SSB_RSRQ_SortedBy_RSRP_5',
       'NR_Scan_SSB_RSRQ_SortedBy_RSRP_6', 'NR_Scan_SSB_SINR_SortedBy_RSRP_0',
       'NR_Scan_SSB_SINR_SortedBy_RSRP_1', 'NR_Scan_SSB_SINR_SortedBy_RSRP_2',
       'NR_Scan_SSB_SINR_SortedBy_RSRP_3', 'NR_Scan_SSB_SINR_SortedBy_RSRP_4',
       'NR_Scan_SSB_SINR_SortedBy_RSRP_5', 'NR_Scan_SSB_SINR_SortedBy_RSRP_6']

    for row in data['Time'].unique():

        filtered_row_last = data[data["Time"] == row][last_known_data_cols]
        last_known_values = filtered_row_last.apply(lambda x: x.dropna().iloc[-1] if not x.dropna().empty else np.nan)
        filtered_row = pd.concat([pd.Series({'Time': row}), last_known_values])
        new_data = pd.concat([new_data, filtered_row.to_frame().T], ignore_index=True)
                
    return new_data

In [74]:
def dl_group_by_time(data):
    new_data = pd.DataFrame() 
        
    last_known_data_cols = ['Longitude', 'Latitude','NR_UE_PCI_0','NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_PCI_1', 
                            'NR_UE_Nbr_PCI_2', 'NR_UE_Nbr_PCI_3', 'NR_UE_Nbr_PCI_4','NR_UE_Timing_Advance', 
                            'NR_UE_Pathloss_DL_0', 'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL', 'NR_UE_NACK_Rate_DL_0',
                            'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_MCS_DL_0', 'NR_UE_RB_Num_DL_0',
                            'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 'NR_UE_BLER_DL_0',
                            'NR_UE_CCE_AggregationLev_0', 'NR_UE_Power_Tx_PUSCH_0',
                            'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0', 'NR_UE_RACH_Attempt',
                            'NR_UE_RACH_OK', 'NR_UE_RACH_Fail', 'NR_UE_RACH_Procedure_Count', 'NR_UE_RRCReEstAttempt', 
                            'NR_UE_RRCReEstFail', 'NR_UE_RRCReEst_EndResult', 'NR_UE_RRCConnectionAttempt',
                            'NR_UE_RRCConnectionSetupOk', 'NR_UE_RRCConnectionComplete',
                            'NR_UE_RRCConnectionDrop', 'NR_UE_RRCHOAttempt', 'NR_UE_RRCHOOK',
                            'NR_RRC_MsgType', 'NAS_5GS_MM_MessageType', 'NAS_5GS_SM_MessageType']

    mean_cols = ['NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0','NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRP_2',
                'NR_UE_Nbr_RSRP_3', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_0','NR_UE_Nbr_RSRQ_1', 'NR_UE_Nbr_RSRQ_2', 'NR_UE_Nbr_RSRQ_3',
                'NR_UE_Nbr_RSRQ_4']

    for row in data['Time'].unique():

        filtered_row_mean = data[data["Time"] == row][mean_cols].mean()
        filtered_row_last = data[data["Time"] == row][last_known_data_cols]
        last_known_values = filtered_row_last.apply(lambda x: x.dropna().iloc[-1] if not x.dropna().empty else np.nan)
        filtered_row = pd.concat([pd.Series({'Time': row}), filtered_row_mean, last_known_values])
        new_data = pd.concat([new_data, filtered_row.to_frame().T], ignore_index=True)
                
    return new_data    

In [75]:
def ul_group_by_time(data):
    new_data = pd.DataFrame() 
        
    last_known_data_cols = ['Longitude', 'Latitude','NR_UE_PCI_0','NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_PCI_1', 
                            'NR_UE_Nbr_PCI_2', 'NR_UE_Nbr_PCI_3', 'NR_UE_Nbr_PCI_4',
                            'NR_UE_Timing_Advance', 'NR_UE_Pathloss_DL_0','NR_UE_Throughput_PDCP_DL', 'NR_UE_NACK_Rate_DL_0',
                            'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_MCS_DL_0', 'NR_UE_RB_Num_DL_0','NR_UE_Modulation_Avg_DL_0', 
                            'NR_UE_RI_DL_0', 'NR_UE_BLER_DL_0','NR_UE_CCE_AggregationLev_0', 'NR_UE_Power_Tx_PUSCH_0',
                            'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0', 'NR_UE_RACH_Attempt',
                            'NR_UE_RACH_OK', 'NR_UE_RACH_Fail', 'NR_UE_RACH_Procedure_Count','NR_UE_RRCReEstAttempt', 
                            'NR_UE_RRCReEstFail','NR_UE_RRCReEst_EndResult', 'NR_UE_RRCConnectionAttempt',
                            'NR_UE_RRCConnectionSetupOk', 'NR_UE_RRCConnectionComplete','NR_UE_RRCConnectionDrop', 
                            'NR_UE_RRCHOAttempt', 'NR_UE_RRCHOOK','NR_RRC_MsgType', 'NAS_5GS_MM_MessageType', 
                            'NR_UE_Throughput_RLC_UL','App_Throughput_UL']

    mean_cols = ['NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0','NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRP_2',
                'NR_UE_Nbr_RSRP_3', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_0','NR_UE_Nbr_RSRQ_1', 'NR_UE_Nbr_RSRQ_2', 'NR_UE_Nbr_RSRQ_3',
                'NR_UE_Nbr_RSRQ_4']

    for row in data['Time'].unique():

        filtered_row_mean = data[data["Time"] == row][mean_cols].mean()
        filtered_row_last = data[data["Time"] == row][last_known_data_cols]
        last_known_values = filtered_row_last.apply(lambda x: x.dropna().iloc[-1] if not x.dropna().empty else np.nan)
        filtered_row = pd.concat([pd.Series({'Time': row}), filtered_row_mean, last_known_values])
        new_data = pd.concat([new_data, filtered_row.to_frame().T], ignore_index=True)
                
    return new_data    

In [77]:
dl_data = dl_group_by_time(dl_data)
ul_data = ul_group_by_time(ul_data)
scanner_data = scanner_group_by_time(scanner_data)

dl_data["Time"] = pd.to_datetime(dl_data["Time"])
ul_data["Time"] = pd.to_datetime(ul_data["Time"])
scanner_data["Time"] = pd.to_datetime(scanner_data["Time"])

dl_data.columns = ["Time" if x== "Time" else "Longitude" if x=="Longitude" else "Latitude" if x=="Latitude" else f"DL_{x}" for x in dl_data.columns]
ul_data.columns = ["Time" if x== "Time" else "Longitude" if x=="Longitude" else "Latitude" if x=="Latitude" else f"UL_{x}" for x in ul_data.columns]
scanner_data.columns = ["Time" if x== "Time" else "Longitude" if x=="Longitude" else "Latitude" if x=="Latitude" else f"SC_{x}" for x in scanner_data.columns]

In [78]:
dl_data.columns

Index(['Time', 'DL_NR_UE_RSRP_0', 'DL_NR_UE_RSRQ_0', 'DL_NR_UE_SINR_0',
       'DL_NR_UE_Nbr_RSRP_0', 'DL_NR_UE_Nbr_RSRP_1', 'DL_NR_UE_Nbr_RSRP_2',
       'DL_NR_UE_Nbr_RSRP_3', 'DL_NR_UE_Nbr_RSRP_4', 'DL_NR_UE_Nbr_RSRQ_0',
       'DL_NR_UE_Nbr_RSRQ_1', 'DL_NR_UE_Nbr_RSRQ_2', 'DL_NR_UE_Nbr_RSRQ_3',
       'DL_NR_UE_Nbr_RSRQ_4', 'Longitude', 'Latitude', 'DL_NR_UE_PCI_0',
       'DL_NR_UE_Nbr_PCI_0', 'DL_NR_UE_Nbr_PCI_1', 'DL_NR_UE_Nbr_PCI_2',
       'DL_NR_UE_Nbr_PCI_3', 'DL_NR_UE_Nbr_PCI_4', 'DL_NR_UE_Timing_Advance',
       'DL_NR_UE_Pathloss_DL_0', 'DL_NR_UE_Throughput_PDCP_DL',
       'DL_App_Throughput_DL', 'DL_NR_UE_NACK_Rate_DL_0',
       'DL_NR_UE_Ack_As_Nack_DL_0', 'DL_NR_UE_MCS_DL_0',
       'DL_NR_UE_RB_Num_DL_0', 'DL_NR_UE_Modulation_Avg_DL_0',
       'DL_NR_UE_RI_DL_0', 'DL_NR_UE_BLER_DL_0',
       'DL_NR_UE_CCE_AggregationLev_0', 'DL_NR_UE_Power_Tx_PUSCH_0',
       'DL_NR_UE_Power_Tx_PRACH_0', 'DL_NR_UE_NACK_Rate_UL_0',
       'DL_NR_UE_RACH_Attempt', 'DL_NR_UE_RACH_OK', '

In [79]:
joined_time_set = set([*dl_data["Time"].unique(),*ul_data["Time"].unique(),*scanner_data["Time"].unique()])
df = pd.DataFrame()

df["Time"] = sorted(list(joined_time_set))
df = df.merge(dl_data, how="left", on="Time")
df = df.merge(ul_data, how="left", on="Time")
df = df.merge(scanner_data, how="left", on="Time")

for col in ["Latitude", "Longitude"]:
    df[col] = (
        df.get(col)
        .combine_first(df.get(f"{col}_x"))
        .combine_first(df.get(f"{col}_y"))
    )
    df = df.drop(columns=[f"{col}_x",f"{col}_y"])


In [80]:
dropped_columns = ["DL_NR_UE_RRCHOAttempt",
    "UL_NR_UE_RRCHOAttempt",
    "UL_NR_UE_RRCHOOK",
    "UL_NR_UE_RACH_Fail",
    "UL_NR_UE_RACH_Procedure_Count", 
    "DL_NR_UE_RACH_Procedure_Count",
    "UL_NR_UE_RRCReEstAttempt",
    "UL_NR_UE_RRCReEstFail", 
    "UL_NR_UE_RRCReEst_EndResult",
    "DL_NR_UE_RRCReEst_EndResult",
    "UL_NR_UE_RRCConnectionAttempt", 
    "UL_NR_UE_RRCConnectionSetupOk", 
    "UL_NR_UE_RRCConnectionComplete", 
    "UL_NR_UE_RRCConnectionDrop",
    "DL_NR_RRC_MsgType", 
    "DL_NAS_5GS_MM_MessageType",
    "DL_NAS_5GS_SM_MessageType",
    "UL_NAS_5GS_MM_MessageType",
    "DL_NR_UE_RACH_Attempt",
    "DL_NR_UE_RACH_OK",
    "DL_NR_UE_RACH_Fail",
    "UL_NR_UE_RACH_Attempt",
    "UL_NR_UE_RACH_OK",
    "DL_NR_UE_RRCReEstAttempt",
    "DL_NR_UE_RRCReEstFail",
    "DL_NR_UE_RRCConnectionAttempt",
    "DL_NR_UE_RRCConnectionSetupOk",
    "DL_NR_UE_RRCConnectionComplete",
    "DL_NR_UE_RRCConnectionDrop",
    "DL_NR_UE_RRCHOOK",
    "UL_NR_RRC_MsgType",
    "SC_NR_Scan_PCI_SortedBy_RSRP_0"
    ]
df = df.drop(columns=dropped_columns)

In [81]:
linear_cols = [
    "DL_NR_UE_Nbr_RSRP_0", "UL_NR_UE_Nbr_RSRP_0",
    "DL_NR_UE_Nbr_RSRQ_0", "UL_NR_UE_Nbr_RSRQ_0",
    "DL_NR_UE_Nbr_RSRP_1", "UL_NR_UE_Nbr_RSRP_1",
    "DL_NR_UE_Nbr_RSRQ_1", "UL_NR_UE_Nbr_RSRQ_1",
    "DL_NR_UE_Nbr_RSRP_2", "UL_NR_UE_Nbr_RSRP_2",
    "DL_NR_UE_Nbr_RSRQ_2", "UL_NR_UE_Nbr_RSRQ_2",
    "DL_NR_UE_Nbr_RSRP_3", "UL_NR_UE_Nbr_RSRP_3",
    "DL_NR_UE_Nbr_RSRQ_3", "UL_NR_UE_Nbr_RSRQ_3",
    "DL_NR_UE_Nbr_RSRP_4", "UL_NR_UE_Nbr_RSRP_4",
    "DL_NR_UE_Nbr_RSRQ_4", "UL_NR_UE_Nbr_RSRQ_4",
    "DL_NR_UE_RSRP_0","UL_NR_UE_RSRP_0",
    "DL_NR_UE_RSRQ_0","UL_NR_UE_RSRQ_0",
    "DL_NR_UE_SINR_0", "UL_NR_UE_SINR_0",
    "DL_NR_UE_Pathloss_DL_0", "UL_NR_UE_Pathloss_DL_0",
    "DL_App_Throughput_DL","UL_App_Throughput_UL",
    "DL_NR_UE_NACK_Rate_DL_0","DL_NR_UE_Ack_As_Nack_DL_0",
    "UL_NR_UE_NACK_Rate_DL_0",
    "UL_NR_UE_Ack_As_Nack_DL_0",
    "UL_NR_UE_Throughput_RLC_UL",
    "UL_NR_UE_Throughput_PDCP_DL",
    "DL_NR_UE_Throughput_PDCP_DL"
    ]

forward_fill_cols = [
    "DL_NR_UE_Timing_Advance", "UL_NR_UE_Timing_Advance",
    "UL_NR_UE_RI_DL_0","DL_NR_UE_RI_DL_0",
    "DL_NR_UE_BLER_DL_0","UL_NR_UE_BLER_DL_0",
    "DL_NR_UE_Power_Tx_PUSCH_0","DL_NR_UE_Power_Tx_PRACH_0",
    "UL_NR_UE_Power_Tx_PUSCH_0","UL_NR_UE_Power_Tx_PRACH_0",
    "UL_NR_UE_NACK_Rate_UL_0","DL_NR_UE_NACK_Rate_UL_0",

    "DL_NR_UE_PCI_0", "UL_NR_UE_PCI_0",    
    "DL_NR_UE_Nbr_PCI_0", "UL_NR_UE_Nbr_PCI_0",
    "DL_NR_UE_Nbr_PCI_1", "UL_NR_UE_Nbr_PCI_1",
    "DL_NR_UE_Nbr_PCI_2", "UL_NR_UE_Nbr_PCI_2",
    "DL_NR_UE_Nbr_PCI_3", "UL_NR_UE_Nbr_PCI_3",
    "DL_NR_UE_Nbr_PCI_4", "UL_NR_UE_Nbr_PCI_4",
    "DL_NR_UE_MCS_DL_0","UL_NR_UE_MCS_DL_0",
    "DL_NR_UE_RB_Num_DL_0","UL_NR_UE_RB_Num_DL_0",
    "DL_NR_UE_Modulation_Avg_DL_0","UL_NR_UE_Modulation_Avg_DL_0",
    "DL_NR_UE_CCE_AggregationLev_0","UL_NR_UE_CCE_AggregationLev_0",
    "SC_NR_Scan_NR_ARFCN",
    "SC_NR_Scan_PCI_SortedBy_RSRP_1",
    "SC_NR_Scan_PCI_SortedBy_RSRP_2",
    "SC_NR_Scan_PCI_SortedBy_RSRP_3",
    "SC_NR_Scan_PCI_SortedBy_RSRP_4",
    "SC_NR_Scan_PCI_SortedBy_RSRP_5",
    "SC_NR_Scan_PCI_SortedBy_RSRP_6",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_0",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_1",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_2",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_3",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_4",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_5",
    "SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_6",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_0",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_1",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_2",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_3",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_4",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_5",
    "SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_6",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_0",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_1",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_2",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_3",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_4",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_5",
    "SC_NR_Scan_SSB_SINR_SortedBy_RSRP_6"
    ]

In [82]:
null_ratios = df.isnull().mean() * 100
for column, null_ratio in null_ratios.items():
    print(f"{column}: {null_ratio:.2f}% null")

Time: 0.00% null
DL_NR_UE_RSRP_0: 90.86% null
DL_NR_UE_RSRQ_0: 90.86% null
DL_NR_UE_SINR_0: 90.89% null
DL_NR_UE_Nbr_RSRP_0: 93.35% null
DL_NR_UE_Nbr_RSRP_1: 97.65% null
DL_NR_UE_Nbr_RSRP_2: 99.40% null
DL_NR_UE_Nbr_RSRP_3: 99.89% null
DL_NR_UE_Nbr_RSRP_4: 99.99% null
DL_NR_UE_Nbr_RSRQ_0: 93.35% null
DL_NR_UE_Nbr_RSRQ_1: 97.65% null
DL_NR_UE_Nbr_RSRQ_2: 99.40% null
DL_NR_UE_Nbr_RSRQ_3: 99.89% null
DL_NR_UE_Nbr_RSRQ_4: 99.99% null
DL_NR_UE_PCI_0: 90.58% null
DL_NR_UE_Nbr_PCI_0: 93.35% null
DL_NR_UE_Nbr_PCI_1: 97.65% null
DL_NR_UE_Nbr_PCI_2: 99.40% null
DL_NR_UE_Nbr_PCI_3: 99.89% null
DL_NR_UE_Nbr_PCI_4: 99.99% null
DL_NR_UE_Timing_Advance: 99.81% null
DL_NR_UE_Pathloss_DL_0: 90.88% null
DL_NR_UE_Throughput_PDCP_DL: 90.65% null
DL_App_Throughput_DL: 95.67% null
DL_NR_UE_NACK_Rate_DL_0: 90.84% null
DL_NR_UE_Ack_As_Nack_DL_0: 90.84% null
DL_NR_UE_MCS_DL_0: 90.84% null
DL_NR_UE_RB_Num_DL_0: 90.84% null
DL_NR_UE_Modulation_Avg_DL_0: 90.84% null
DL_NR_UE_RI_DL_0: 87.72% null
DL_NR_UE_BLER_DL_

In [83]:
for col in forward_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(method='ffill')
    else: print(f"{col} not found")

for col in forward_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(method='bfill')
    else: print(f"{col} not found")

for col in linear_cols:
    if col in df.columns:
        #print(df[col])
        #df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].interpolate(method='linear', limit_direction='both')
    else: print(f"{col} not found")
    
for col in linear_cols:
    if col in df.columns:
        df[col] = df[col].fillna(method='ffill')
    else: print(f"{col} not found")

for col in linear_cols:
    if col in df.columns:
        df[col] = df[col].fillna(method='bfill')
    else: print(f"{col} not found")


C:\Users\Kutay\AppData\Local\Temp\ipykernel_1196\222765641.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill')
C:\Users\Kutay\AppData\Local\Temp\ipykernel_1196\222765641.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(method='ffill')
C:\Users\Kutay\AppData\Local\Temp\ipykernel_1196\222765641.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='bfill')
C:\Users\Kutay\AppData\Local\Temp\ipykernel_1196\222765641.py:15: FutureWarning: Series.interpolate with object dtype is deprecated and will raise

In [84]:
drop_row_cols = [
    'Longitude', 'Latitude'
]
df = df.dropna(subset=drop_row_cols)

In [85]:
for column in df.columns:
    null_ratio = df[column].isnull().mean() * 100 
    print(f"{column}: {null_ratio:.2f}% null")

Time: 0.00% null
DL_NR_UE_RSRP_0: 0.00% null
DL_NR_UE_RSRQ_0: 0.00% null
DL_NR_UE_SINR_0: 0.00% null
DL_NR_UE_Nbr_RSRP_0: 0.00% null
DL_NR_UE_Nbr_RSRP_1: 0.00% null
DL_NR_UE_Nbr_RSRP_2: 0.00% null
DL_NR_UE_Nbr_RSRP_3: 0.00% null
DL_NR_UE_Nbr_RSRP_4: 0.00% null
DL_NR_UE_Nbr_RSRQ_0: 0.00% null
DL_NR_UE_Nbr_RSRQ_1: 0.00% null
DL_NR_UE_Nbr_RSRQ_2: 0.00% null
DL_NR_UE_Nbr_RSRQ_3: 0.00% null
DL_NR_UE_Nbr_RSRQ_4: 0.00% null
DL_NR_UE_PCI_0: 0.00% null
DL_NR_UE_Nbr_PCI_0: 0.00% null
DL_NR_UE_Nbr_PCI_1: 0.00% null
DL_NR_UE_Nbr_PCI_2: 0.00% null
DL_NR_UE_Nbr_PCI_3: 0.00% null
DL_NR_UE_Nbr_PCI_4: 0.00% null
DL_NR_UE_Timing_Advance: 0.00% null
DL_NR_UE_Pathloss_DL_0: 0.00% null
DL_NR_UE_Throughput_PDCP_DL: 0.00% null
DL_App_Throughput_DL: 0.00% null
DL_NR_UE_NACK_Rate_DL_0: 0.00% null
DL_NR_UE_Ack_As_Nack_DL_0: 0.00% null
DL_NR_UE_MCS_DL_0: 0.00% null
DL_NR_UE_RB_Num_DL_0: 0.00% null
DL_NR_UE_Modulation_Avg_DL_0: 0.00% null
DL_NR_UE_RI_DL_0: 0.00% null
DL_NR_UE_BLER_DL_0: 0.00% null
DL_NR_UE_CCE_Ag

In [86]:
df = pd.get_dummies(df, columns=['DL_NR_UE_PCI_0', 'DL_NR_UE_Nbr_PCI_0', 'DL_NR_UE_Nbr_PCI_1', 'DL_NR_UE_Nbr_PCI_2', 'DL_NR_UE_Nbr_PCI_3', 'DL_NR_UE_Nbr_PCI_4','DL_NR_UE_MCS_DL_0', 'DL_NR_UE_RI_DL_0', 'DL_NR_UE_CCE_AggregationLev_0',
                                 'UL_NR_UE_PCI_0', 'UL_NR_UE_Nbr_PCI_0', 'UL_NR_UE_Nbr_PCI_1', 'UL_NR_UE_Nbr_PCI_2', 'UL_NR_UE_Nbr_PCI_3', 'UL_NR_UE_Nbr_PCI_4','UL_NR_UE_MCS_DL_0', 'UL_NR_UE_RI_DL_0', 'UL_NR_UE_CCE_AggregationLev_0',
                                 "DL_NR_UE_Modulation_Avg_DL_0","UL_NR_UE_Modulation_Avg_DL_0"], dummy_na=False)

In [87]:
for column in df.columns:
    null_ratio = df[column].isnull().mean() * 100 
    print(f"{column}: {null_ratio:.2f}% null")

Time: 0.00% null
DL_NR_UE_RSRP_0: 0.00% null
DL_NR_UE_RSRQ_0: 0.00% null
DL_NR_UE_SINR_0: 0.00% null
DL_NR_UE_Nbr_RSRP_0: 0.00% null
DL_NR_UE_Nbr_RSRP_1: 0.00% null
DL_NR_UE_Nbr_RSRP_2: 0.00% null
DL_NR_UE_Nbr_RSRP_3: 0.00% null
DL_NR_UE_Nbr_RSRP_4: 0.00% null
DL_NR_UE_Nbr_RSRQ_0: 0.00% null
DL_NR_UE_Nbr_RSRQ_1: 0.00% null
DL_NR_UE_Nbr_RSRQ_2: 0.00% null
DL_NR_UE_Nbr_RSRQ_3: 0.00% null
DL_NR_UE_Nbr_RSRQ_4: 0.00% null
DL_NR_UE_Timing_Advance: 0.00% null
DL_NR_UE_Pathloss_DL_0: 0.00% null
DL_NR_UE_Throughput_PDCP_DL: 0.00% null
DL_App_Throughput_DL: 0.00% null
DL_NR_UE_NACK_Rate_DL_0: 0.00% null
DL_NR_UE_Ack_As_Nack_DL_0: 0.00% null
DL_NR_UE_RB_Num_DL_0: 0.00% null
DL_NR_UE_BLER_DL_0: 0.00% null
DL_NR_UE_Power_Tx_PUSCH_0: 0.00% null
DL_NR_UE_Power_Tx_PRACH_0: 0.00% null
DL_NR_UE_NACK_Rate_UL_0: 0.00% null
UL_NR_UE_RSRP_0: 0.00% null
UL_NR_UE_RSRQ_0: 0.00% null
UL_NR_UE_SINR_0: 0.00% null
UL_NR_UE_Nbr_RSRP_0: 0.00% null
UL_NR_UE_Nbr_RSRP_1: 0.00% null
UL_NR_UE_Nbr_RSRP_2: 0.00% null
UL_NR

In [89]:
df.to_csv("5G_DataSet.csv")